# Наследие Пскова: расчёт и карты

Запуск из корня проекта или каталога `notebooks`. Путь к внешним данным можно задать переменными `PSKOKN_DIR` и `PSKOV_BLOCKSNET_DIR`. Входы, даты, единицы учёта и ограничения описаны в каталоге источников. Выходы — `data/processed` и `maps` в рабочей папке проекта.

In [ ]:
import os
from pathlib import Path

if Path.cwd().name == "notebooks":
    os.chdir("..")
assert Path("docs/research_brief.md").is_file(), "Откройте ноутбук в каталоге проекта"

import geopandas as gpd
import numpy as np
import pandas as pd
from PIL import Image
from scripts.prepare_heritage import BLOCKSNET, PSKOKN, run_pipeline
from scripts.plot_heritage import build_maps

pskokn = Path(os.environ.get("PSKOKN_DIR", PSKOKN))
blocksnet = Path(os.environ.get("PSKOV_BLOCKSNET_DIR", BLOCKSNET))
processed = Path("data/processed")
maps = Path("maps")
from scripts.plot_pokrova_boundaries import draw_boundaries
from scripts.build_research_pdf import build_pdf

## Отбор по границе городского округа

Срезы сохраняются раздельно. Количество строк GeoPackage измеряет число геометрических записей, а количество номеров в локальной выгрузке ЕГРОКН зависит от покрытия и наличия координат.

In [ ]:
summary = run_pipeline(processed, pskokn, blocksnet)
selection = pd.read_csv(processed / "spatial_selection.csv")
selection

In [ ]:
gpkg = processed / "heritage.gpkg"
points = gpd.read_file(gpkg, layer="egrokn_points")
city = gpd.read_file(gpkg, layer="egrokn_points_city")
assert len(points) == summary["egrokn"]["with_coordinates"] == 1168
assert len(city) == summary["egrokn"]["city_points"] == 301
assert points.crs.to_epsg() == city.crs.to_epsg() == 4326
assert points.geometry.notna().all() and points.geometry.is_valid.all()
assert points.record_uid.is_unique
assert np.allclose(points.geometry.x, points.longitude, rtol=0, atol=1e-10)
assert np.allclose(points.geometry.y, points.latitude, rtol=0, atol=1e-10)
assert set(city.record_uid) == set(points.loc[points.in_city_intersects, "record_uid"])

comparison = pd.read_csv(processed / "blocksnet_comparison.csv")
assert comparison.row_count_matches.all()
assert comparison.aligned_geometry_within_1m.equals(comparison.new_selection_rows)
comparison

## Карты и сценарные точки

Компоненты ЮНЕСКО, объекты, подложка OSM и сценарные точки имеют разные источники. Для Покровского угла пространственное применение режимов требует утверждённой геометрии границ и отдельной проверки.

In [ ]:
map_check = build_maps(pskokn, processed, maps)
checks = draw_boundaries(
    maps / "04_охранные_границы.png", pskokn,
    maps / "03_точки_фломастер_плана.geojson", processed / "pokrova_site_screening.csv",
)
assert max(checks.values()) < 15
assert round(map_check["walk_length_m"]) == 245
png_files = sorted(maps.glob("0*.png"))
assert len(png_files) == 4
for path in png_files:
    with Image.open(path) as image:
        assert image.size == (2000, 1400)
sites = gpd.read_file(maps / "03_точки_фломастер_плана.geojson")
assert set(sites.scenario_id) == {"S1", "S2"}
assert sites.geometry.is_valid.all()
assert sites.related_object_id.eq("okn:601510242910006").all()
map_check

## Самостоятельный отчёт

Карты, выводы, условия сценария и источники собраны в одном PDF.

In [ ]:
assert build_pdf() == 8
Path("output/pdf/Наследие_Пскова.pdf")